In [0]:
import os, sys, importlib, glob
from collections import defaultdict
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(os.path.join(repo_root, "src"))
import entsoe_parser
importlib.reload(entsoe_parser)
from entsoe_parser import parse

path = "/Volumes/power_prices/bronze/raw_files/day_ahead_price/DE_LU/2025/day_ahead_price_DE_LU_202506012200_202506022200.xml"
with open(path, "rb") as f:
    rows = parse(f.read())

print(f"File size: {os.path.getsize(path):,} bytes | parsed rows: {len(rows)}")

# One line per TimeSeries/Period: what the document actually contains
groups = defaultdict(list)
for r in rows:
    groups[(r["series_mrid"], r["business_type"], r["resolution"], r["curve_type"], r["period_start"])].append(r)

print(f"\n{'series':<7} {'business':<9} {'resolution':<11} {'curve':<6} {'period start (UTC)':<26} {'points':>6} {'filled':>6} {'min':>8} {'max':>8}")
for (mrid, bt, res, curve, start), rs in sorted(groups.items(), key=lambda kv: (kv[0][4], kv[0][2])):
    vals = [r["value"] for r in rs]
    print(f"{mrid:<7} {bt or '':<9} {res:<11} {curve:<6} {str(start):<26} {len(rs):>6} "
          f"{sum(r['is_filled'] for r in rs):>6} {min(vals):>8.2f} {max(vals):>8.2f}")

print("\nUnits:", {(r["unit"], r["currency"]) for r in rows})
print("Zones (in/out):", {(r["in_domain"], r["out_domain"]) for r in rows})

display(pd.DataFrame(rows)[["series_mrid", "resolution", "position", "timestamp_utc", "value", "is_filled"]].head(12))

In [0]:
import silver_loader
importlib.reload(silver_loader)

print("Silver load:", silver_loader.run(spark, ["day_ahead_price"]))

check = spark.sql("""
    SELECT COUNT(*) AS hours,
           ROUND(MIN(price_eur_mwh), 2) AS min_price,
           ROUND(MAX(price_eur_mwh), 2) AS max_price,
           CONCAT_WS(',', COLLECT_SET(source_resolution)) AS resolutions,
           MIN(local_hour) AS first_hour, MAX(local_hour) AS last_hour
    FROM power_prices.silver.price
    WHERE zone_code = 'DE_LU' AND local_date = '2025-06-02'
""").collect()[0]
print(check)

assert check.hours == 24 and check.resolutions == "PT60M", "Expected 24 hourly rows from the PT60M series"
assert check.min_price == 11.00 and check.max_price == 245.37, "Values should match the hourly auction series"
print("Price rule passed: hourly auction series used, 15-minute product ignored")